In [1]:
"""measure_gp_runtime.py

Quick‑and‑dirty benchmark script (no `main()` wrapper) for the GPFlow models
trained with **n = 400** and **noise = 0.0**.  As soon as you run the file in
VS Code, it:

1. Loads the pickled model dictionary (``gp_models_absolute.pkl``).
2. Reads the *single* test CSV with noise 0.0.
3. Times **one full batch call** to ``model.predict_f`` for each matching model
   (i.e. all seeds trained with that config).  ‑‑> *This is the runtime for the
   **entire** test‑set, not per‑sample.*  A per‑sample estimate is printed too.
4. Optionally writes a CSV with the timing metadata only (no predictions).

Change the constants at the top if your paths differ.  Toggle ``SAVE_CSV`` to
False if you don’t want the output file.
"""

from __future__ import annotations

import pickle
import time
from pathlib import Path

import gpflow  # noqa: F401 (needed for unpickling GP models)
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------------------------------------------
# Configuration ------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
BASE_DIR = Path("3D-Data")                   # folder with CSV data
MODEL_PATH = Path("gp_models_absolute.pkl")  # pickled dict of GPFlow models
SCALER_PATH = Path("scaler_absolute.pkl")    # fitted StandardScaler (optional)

CATEGORY = "Measured_Normalized"             # sub‑folder for the CSVs
TRAIN_N = 400                                # we only benchmark models with this n
NOISE_TRAIN = 0.0                            # training noise level to filter on
TEST_NOISE = 0.0                             # test CSV noise level
SAVE_CSV = True                              # toggle CSV export
CSV_PATH = Path("gp_runtime_results.csv")

# --------------------------------------------------------------------------------------
# Helper -------------------------------------------------------------------------------
# --------------------------------------------------------------------------------------

def load_test_df() -> pd.DataFrame:
    """Load the test CSV matching *TEST_NOISE*."""
    fp = BASE_DIR / CATEGORY / f"gnd_n={50}_noise={TEST_NOISE}.csv"  # <<< adjust if gnd size differs
    if not fp.exists():
        raise FileNotFoundError(f"Test CSV not found: {fp}")
    return pd.read_csv(fp)


def predict_in_memory(model, test_df: pd.DataFrame, scaler: StandardScaler | None):
    """Run GP inference and (optionally) inverse‑transform outputs."""
    outs = ["X", "Y", "Z"]
    rss = test_df.drop(columns=outs).values
    mean, _ = model.predict_f(rss)

    if scaler is None:
        return mean.numpy()

    zeros = np.zeros_like(rss)
    merged = np.hstack([zeros, mean])
    return scaler.inverse_transform(merged)[:, -3:]


# --------------------------------------------------------------------------------------
# Benchmark ---------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
print("Loading resources …", flush=True)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Could not locate {MODEL_PATH}")

with MODEL_PATH.open("rb") as fh:
    gp_models = pickle.load(fh)

TEST_DF = load_test_df()
N_TEST = len(TEST_DF)

SCALER = None
if SCALER_PATH.exists():
    with SCALER_PATH.open("rb") as fh:
        SCALER = pickle.load(fh)

print(f"Benchmarking models with n={TRAIN_N}, noise=0.0 on {N_TEST} samples\n")

timings: list[dict] = []

for (cat, tr_noise, n_train, seed), model in gp_models.items():
    if cat != CATEGORY or tr_noise != NOISE_TRAIN or n_train != TRAIN_N:
        continue  # skip models outside our filter

    t0 = time.perf_counter()
    _ = predict_in_memory(model, TEST_DF, SCALER)
    dt = time.perf_counter() - t0

    timings.append(
        {
            "seed": seed,
            "runtime_s": dt,
            "per_sample_s": dt / N_TEST,
            "n_test": N_TEST,
        }
    )

    print(
        f"Seed {seed:2d} | {N_TEST} predictions | total: {dt:.4f} s | per‑sample: {dt/N_TEST:.6f} s"
    )

if not timings:
    print("⚠️  No models matched the filter criteria!")
else:
    # Build DataFrame and optionally save
    results_df = pd.DataFrame(timings)

    if SAVE_CSV:
        results_df.to_csv(CSV_PATH, index=False)
        print(f"\nTiming table written to → {CSV_PATH.resolve()}")

print("\nDone.")


2025-05-14 10:25:33.747966: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-14 10:25:33.748007: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-14 10:25:33.749293: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-14 10:25:33.756529: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-14 10:25:36.325897: W tensorflow/compiler/tf2

Loading resources …


2025-05-14 10:25:43.732334: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
/home/sumo/anikraft/miniconda3/envs/trieste_env/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Benchmarking models with n=400, noise=0.0 on 125000 samples

Seed  0 | 125000 predictions | total: 3.7716 s | per‑sample: 0.000030 s
Seed  1 | 125000 predictions | total: 3.7822 s | per‑sample: 0.000030 s
Seed  2 | 125000 predictions | total: 3.8970 s | per‑sample: 0.000031 s
Seed  3 | 125000 predictions | total: 3.8163 s | per‑sample: 0.000031 s
Seed  4 | 125000 predictions | total: 3.7169 s | per‑sample: 0.000030 s
Seed  5 | 125000 predictions | total: 3.6517 s | per‑sample: 0.000029 s
Seed  6 | 125000 predictions | total: 3.7221 s | per‑sample: 0.000030 s
Seed  7 | 125000 predictions | total: 3.7097 s | per‑sample: 0.000030 s
Seed  8 | 125000 predictions | total: 3.7063 s | per‑sample: 0.000030 s
Seed  9 | 125000 predictions | total: 3.7933 s | per‑sample: 0.000030 s

Timing table written to → /home/sumo/anikraft/RSS/chapter4_recreating_results/gp_runtime_results.csv

Done.


In [2]:
"""measure_gp_runtime.py – *relative* models

Benchmarks the inference runtime for **GPFlow models trained on “relative” RSS
features** (10‑dim feature vectors like RSS0/RSS1, …).  Configuration:

* **Training filter** n = 400, noise = 0.0, category = `Measured_relative_Normalized`.
* **Test set** `relative_gnd_n=50_noise=0.0.csv` (exactly 50 ↔ never depends on
  the training sample size).
* **Models file** `gp_models_relative.pkl` (produced by your relative training
  script).
* **Scaler** `scaler_relative.pkl`.

The stopwatch surrounds a single batch call to `model.predict_f`, i.e. the time
reported is for **all 50 × 2500 = 125 000 predictions** in the ground‑truth
file.  A per‑sample estimate (total / N_test) is printed for convenience.

Nothing is written except, optionally, a CSV containing only timing metadata.
Toggle `SAVE_CSV` below to suit your needs.
"""

from __future__ import annotations

import pickle
import time
from pathlib import Path

import gpflow  # noqa: F401 (needed to unpickle GP models)
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------------------------------------------
# Configuration ------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
BASE_DIR = Path("3D-Data")
MODEL_PATH = Path("gp_models_relative.pkl")
SCALER_PATH = Path("scaler_relative.pkl")

CATEGORY = "Measured_relative_Normalized"  # folder containing the relative CSVs
TRAIN_N = 400                              # filter: only models trained on 400 pts
NOISE_TRAIN = 0.0                          # filter: only 0‑noise training
TEST_NOISE = 0.0                           # we benchmark on the 0‑noise test set
GND_N = 50                                 # *fixed* rows in every gnd CSV

SAVE_CSV = True                            # toggle export
CSV_PATH = Path("gp_runtime_results_relative.csv")

# --------------------------------------------------------------------------------------
# Helper -------------------------------------------------------------------------------
# --------------------------------------------------------------------------------------

def load_test_df() -> pd.DataFrame:
    """Return the ground‑truth DataFrame for benchmarking (relative features)."""
    fp = BASE_DIR / CATEGORY / f"relative_gnd_n={GND_N}_noise={TEST_NOISE}.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Test CSV not found: {fp}")
    return pd.read_csv(fp)


def predict_in_memory(model, test_df: pd.DataFrame, scaler: StandardScaler | None):
    """Run model.predict_f and (optionally) inverse‑scale outputs."""
    outs = ["X", "Y", "Z"]
    rss = test_df.drop(columns=outs).values  # 10‑dim relative features
    mean, _ = model.predict_f(rss)

    if scaler is None:
        return mean.numpy()

    zeros = np.zeros_like(rss)
    merged = np.hstack([zeros, mean])
    return scaler.inverse_transform(merged)[:, -3:]


# --------------------------------------------------------------------------------------
# Benchmark ---------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
print("Loading resources …", flush=True)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Could not locate {MODEL_PATH}")

with MODEL_PATH.open("rb") as fh:
    gp_models = pickle.load(fh)

TEST_DF = load_test_df()
N_TEST = len(TEST_DF)

SCALER = None
if SCALER_PATH.exists():
    with SCALER_PATH.open("rb") as fh:
        SCALER = pickle.load(fh)

print(
    f"Benchmarking *relative* models with n={TRAIN_N}, noise=0.0 on {N_TEST} samples\n"
)

timings: list[dict] = []

for (cat, tr_noise, n_train, seed), model in gp_models.items():
    if cat != CATEGORY or tr_noise != NOISE_TRAIN or n_train != TRAIN_N:
        continue  # skip non‑matching models

    start = time.perf_counter()
    _ = predict_in_memory(model, TEST_DF, SCALER)
    dt = time.perf_counter() - start

    timings.append(
        {
            "seed": seed,
            "runtime_s": dt,
            "per_sample_s": dt / N_TEST,
            "n_test": N_TEST,
        }
    )

    print(
        f"Seed {seed:2d} | {N_TEST} predictions | total: {dt:.4f} s | per‑sample: {dt/N_TEST:.6f} s"
    )

if not timings:
    print("⚠️  No models matched the filter criteria!")
else:
    df_results = pd.DataFrame(timings)
    if SAVE_CSV:
        df_results.to_csv(CSV_PATH, index=False)
        print(f"\nTiming table written to → {CSV_PATH.resolve()}")

print("\nDone.")


Loading resources …


/home/sumo/anikraft/miniconda3/envs/trieste_env/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Benchmarking *relative* models with n=400, noise=0.0 on 125000 samples

Seed  0 | 125000 predictions | total: 3.6860 s | per‑sample: 0.000029 s
Seed  1 | 125000 predictions | total: 3.6679 s | per‑sample: 0.000029 s
Seed  2 | 125000 predictions | total: 3.6298 s | per‑sample: 0.000029 s
Seed  3 | 125000 predictions | total: 3.8117 s | per‑sample: 0.000030 s
Seed  4 | 125000 predictions | total: 3.6424 s | per‑sample: 0.000029 s
Seed  5 | 125000 predictions | total: 3.6334 s | per‑sample: 0.000029 s
Seed  6 | 125000 predictions | total: 3.5881 s | per‑sample: 0.000029 s
Seed  7 | 125000 predictions | total: 3.5537 s | per‑sample: 0.000028 s
Seed  8 | 125000 predictions | total: 3.6097 s | per‑sample: 0.000029 s
Seed  9 | 125000 predictions | total: 3.5905 s | per‑sample: 0.000029 s

Timing table written to → /home/sumo/anikraft/RSS/chapter4_recreating_results/gp_runtime_results_relative.csv

Done.


In [3]:
"""measure_gp_runtime_logd.py – *log‑distance* models

Benchmarks the end‑to‑end inference runtime for **GPFlow models that predict
log‑distances (D0..D4) and then perform trilateration** to recover (X, Y, Z).

The stopwatch encloses **all three stages**:

1. `model.predict_f` (log‑distance batch prediction)
2. Post‑processing  (descaling → exponentiation to linear metres)
3. Trilateration   (non‑linear least‑squares for each sample)

Configuration
-------------
* **Training filter** n = 400, noise = 0.0, category =`Measured_log_d_Normalized`.
* **Test set** `gnd_n=50_noise=0.0.csv` (always 50 rows × 2500 = 125 000 samples).
* **Models file** `gp_models_log_d.pkl`.
* **Scaler** `scaler_log_d.pkl`.
* **Beacon layout** [[ 0 ,  0 , 5], [‑2,‑2,5], [‑2, 2,5], [ 2,‑2,5], [ 2, 2,5]].

Only timing metadata may be written (CSV). **No predictions are saved.**
"""

from __future__ import annotations

import pickle
import time
from pathlib import Path

import gpflow  # noqa: F401 (for unpickling)
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------------------------------------------
# Configuration ------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
BASE_DIR = Path("3D-Data")
MODEL_PATH = Path("gp_models_log_d.pkl")
SCALER_PATH = Path("scaler_log_d.pkl")

CATEGORY = "Measured_log_d_Normalized"  # folder with log‑d CSVs
TRAIN_N = 400                            # filter: only models trained on 400 rows
NOISE_TRAIN = 0.0                        # filter: only 0‑noise training data
TEST_NOISE = 0.0                         # benchmark on 0‑noise test set
GND_N = 50                               # fixed rows in every gnd CSV

SAVE_CSV = True                          # toggle export
CSV_PATH = Path("gp_runtime_results_logd.csv")

# --------------------------------------------------------------------------------------
# Trilateration helpers ----------------------------------------------------------------
# --------------------------------------------------------------------------------------
BEACON_COORDS = np.array(
    [
        [0, 0, 5],
        [-2, -2, 5],
        [-2,  2, 5],
        [2, -2, 5],
        [2,  2, 5],
    ]
)


def mse_error(pos, beacon_coords, distances):
    pred = np.linalg.norm(beacon_coords - pos, axis=1)
    return np.mean((pred - distances) ** 2)


def trilaterate_mse(distances, beacon_coords=BEACON_COORDS):
    guess = np.mean(beacon_coords, axis=0)
    bounds = [(None, None), (None, None), (None, 5)]  # z limited to [0,5]
    res = minimize(mse_error, guess, args=(beacon_coords, distances), method="L-BFGS-B", bounds=bounds)
    if not res.success:
        raise ValueError("Trilateration failed: " + res.message)
    return res.x

# --------------------------------------------------------------------------------------
# Helper functions ---------------------------------------------------------------------
# --------------------------------------------------------------------------------------

def load_test_df() -> pd.DataFrame:
    fp = BASE_DIR / CATEGORY / f"gnd_n={GND_N}_noise={TEST_NOISE}.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Test CSV not found: {fp}")
    return pd.read_csv(fp)


def predict_trilaterate(
    model,
    test_df: pd.DataFrame,
    scaler: StandardScaler | None = None,
):
    """Full pipeline: predict log‑d, inverse‑scale, exp, trilaterate → (X,Y,Z)."""
    dist_cols = ["D0", "D1", "D2", "D3", "D4"]
    rss = test_df.drop(columns=dist_cols).values  # inputs are RSS0..RSS4 etc.

    # 1) GP prediction (log‑distances)
    mean, _ = model.predict_f(rss)

    # 2) Descale if scaler provided
    if scaler is not None:
        zeros = np.zeros_like(rss)
        mean = scaler.inverse_transform(np.hstack([zeros, mean]))[:, -5:]

    # 3) Exponentiate → linear metres
    lin_dist = np.exp(mean)

    # 4) Trilaterate each sample
    coords = np.empty((lin_dist.shape[0], 3))
    for i, dists in enumerate(lin_dist):
        coords[i] = trilaterate_mse(dists)

    return coords

# --------------------------------------------------------------------------------------
# Benchmark ---------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
print("Loading resources …", flush=True)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Could not locate {MODEL_PATH}")

with MODEL_PATH.open("rb") as fh:
    gp_models = pickle.load(fh)

TEST_DF = load_test_df()
N_TEST = len(TEST_DF)

SCALER = None
if SCALER_PATH.exists():
    with SCALER_PATH.open("rb") as fh:
        SCALER = pickle.load(fh)

print(
    f"Benchmarking *log‑d* models with n={TRAIN_N}, noise=0.0 on {N_TEST} samples\n"
)

timings: list[dict] = []

for (cat, tr_noise, n_train, seed), model in gp_models.items():
    if cat != CATEGORY or tr_noise != NOISE_TRAIN or n_train != TRAIN_N:
        continue

    t0 = time.perf_counter()
    _ = predict_trilaterate(model, TEST_DF, SCALER)
    dt = time.perf_counter() - t0

    timings.append(
        {
            "seed": seed,
            "runtime_s": dt,
            "per_sample_s": dt / N_TEST,
            "n_test": N_TEST,
        }
    )

    print(
        f"Seed {seed:2d} | {N_TEST} predictions | total: {dt:.4f} s | per‑sample: {dt/N_TEST:.6f} s"
    )

if not timings:
    print("⚠️  No models matched the filter criteria!")
else:
    results_df = pd.DataFrame(timings)
    if SAVE_CSV:
        results_df.to_csv(CSV_PATH, index=False)
        print(f"\nTiming table written to → {CSV_PATH.resolve()}")

print("\nDone.")


Loading resources …


/home/sumo/anikraft/miniconda3/envs/trieste_env/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Benchmarking *log‑d* models with n=400, noise=0.0 on 125000 samples

Seed  0 | 125000 predictions | total: 682.9087 s | per‑sample: 0.005463 s

Timing table written to → /home/sumo/anikraft/RSS/chapter4_recreating_results/gp_runtime_results_logd.csv

Done.
